# Final-Final Run — Crystal-Rotation SIIMPL + Directional Head (A100)

Trains `train_gvp_egnn_v22.py` on the **crystal-rotation** (`rotate_crystal_mode=True`) dataset: grazing-free, crystal-frame, corrected Pool B. Includes the Tier-0 recipe fixes (RNG seed, Stage-2 LR 5e-5→3e-4, sign-aware aux annealing).

**Before you start — upload these two files to Google Drive** (folder `MyDrive/siimpl_rot/`):
- `siimpl_train.csv`  (merged Pool A + Pool B, ~2.5 GB)  — local path `C:\Inverse ML\data\siimpl_rot\siimpl_train.csv`
- `siimpl_eval_merged.csv`  (~0.42 GB)  — local path `C:\Inverse ML\data\siimpl_rot\siimpl_eval_merged.csv`

The notebook clones the code (branch `siimpl/v22`), copies the CSVs into place, and launches training. Colab preprocesses the raw CSVs on the first run (builds the kNN/phys cache).

**Runtime:** set to **A100 GPU** (Runtime → Change runtime type → A100).

In [ ]:
# 1. Confirm A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone the repo (branch siimpl/v22) + install deps
#    If the repo is private, set GH_TOKEN below (Colab: use a Secret named GH_TOKEN).
import os
GH_TOKEN = os.environ.get('GH_TOKEN', '')  # leave '' if the repo is public
REPO = 'github.com/cbharathulwar/sbi-srim.git'
url = f'https://{GH_TOKEN + "@" if GH_TOKEN else ""}{REPO}'
%cd /content
!rm -rf sbi-srim
!git clone --branch siimpl/v22 --single-branch $url sbi-srim
%cd /content/sbi-srim
!git log --oneline -1
!pip -q install sbi scipy scikit-learn tensorboard 2>/dev/null; echo done

In [ ]:
# 3. Mount Drive and stage the data into the path v22 expects (data/siimpl_rot/)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
SRC = '/content/drive/MyDrive/siimpl_rot'          # <-- where you uploaded the CSVs
DST = '/content/sbi-srim/data/siimpl_rot'
os.makedirs(DST, exist_ok=True)
for f in ['siimpl_train.csv', 'siimpl_eval_merged.csv']:
    s = os.path.join(SRC, f)
    assert os.path.exists(s), f'MISSING in Drive: {s}  (upload it to {SRC})'
    print('copying', f, f'({os.path.getsize(s)/1e9:.2f} GB) ...')
    shutil.copy(s, os.path.join(DST, f))
!ls -la /content/sbi-srim/data/siimpl_rot/
!head -1 /content/sbi-srim/data/siimpl_rot/siimpl_train.csv

## 4. Train (headline config)

Runs the validated architecture (GVP-EGNN hidden 96 / 6 layers) + Tier-0 fixes on the crystal-rotation data. First run preprocesses the CSVs (builds caches), then trains Stage 1 → Stage 2 → evaluates.

If the Colab session drops mid-run, re-run this cell with `--resume` appended to continue from the last checkpoint.

In [ ]:
%cd /content/sbi-srim
%env KMP_DUPLICATE_LIB_OK=TRUE
%env PYTHONIOENCODING=utf-8
!python src/scripts/train_gvp_egnn_v22.py --directional-head --tag ROT_FINAL

In [ ]:
# 5. Save the trained model + eval results back to Drive
import glob, shutil, os
OUT = '/content/drive/MyDrive/siimpl_rot/ROT_FINAL_outputs'
os.makedirs(OUT, exist_ok=True)
res = '/content/sbi-srim/results/gvp_egnn_v3_siimpl_ROT_FINAL'
for p in glob.glob(res + '/*.pt') + glob.glob(res + '/*.csv') + glob.glob(res + '/*.json'):
    shutil.copy(p, OUT); print('saved', os.path.basename(p))
print('\nOutputs in Drive:', OUT)
!ls -la $OUT

## 6. (Optional) Larger-backbone variant — second run

A size-bumped experiment (hidden 96→128, 6→8 layers). Honest expectation from the codebase review: the task is **information-limited**, so this is unlikely to move the *bulk* median much — the potential upside is in the hard channeling / low-energy tracks. Cheap on A100; run it as a **separate** run for clean attribution.

To enable, edit `src/scripts/train_gvp_egnn_v22.py` at the top: `HIDDEN_DIM = 128`, `N_LAYERS = 8`, then run the cell below with a distinct tag.

In [ ]:
# (only after editing HIDDEN_DIM/N_LAYERS as noted above)
# !python src/scripts/train_gvp_egnn_v22.py --directional-head --tag ROT_BIG